# Data Exploration & Feature Selection

## Overview

This notebook performs comprehensive feature analysis and selection for the reinforcement learning portfolio management system. The goal is to identify the most informative features while reducing dimensionality to improve model training efficiency and reduce overfitting.

## Methodology

The analysis follows a multi-stage approach:

1. **Data Loading & Feature Engineering**: Load weekly market data and engineer technical indicators, rolling statistics, and regime features
2. **Feature Statistics**: Analyze feature quality, variance, and data completeness
3. **Correlation Analysis**: Identify highly correlated feature pairs (|r| > 0.95) to detect redundancy
4. **Principal Component Analysis (PCA)**: Determine the number of components needed to explain variance thresholds (90%, 95%, 99%)
5. **SHAP Analysis**: Use SHAP values from a Random Forest surrogate model to rank feature importance for predicting future returns
6. **Feature Selection**: Combine methods to select 30-40 core features ensuring diversity across feature categories
7. **Performance Validation**: Compare model performance with full vs. selected feature sets
8. **Export**: Save selected features for use in the main portfolio system

## Key Assumptions

- **Target Variable**: Future 1-week log returns (calculated from close prices)
- **Correlation Threshold**: Features with |correlation| > 0.95 are considered redundant
- **PCA Variance Thresholds**: 90%, 95%, and 99% explained variance benchmarks
- **Feature Selection Target**: 30-40 features to balance information content and model complexity


## 1. Setup and Data Loading

### Purpose

This section initializes the data pipeline and loads weekly market data with engineered features. The pipeline handles:
- Weekly data aggregation from daily OHLCV data
- Feature engineering (technical indicators, rolling statistics, regime features)
- Data quality checks and cleaning (NaN handling, infinite value removal)
- Feature filtering based on selected features (if available)

### Key Components

- **SystemConfig**: Configuration object containing all system parameters (tickers, date ranges, feature lookbacks, etc.)
- **DataPipeline**: Handles data loading, feature engineering, and train/val/test splits
- **Feature Engineering**: Creates 35+ features including returns, volatility, technical indicators, and market regime features

### Output

- `featured_data`: DataFrame with all engineered features
- `feature_cols`: List of feature column names (excluding metadata columns like date, ticker, OHLCV)


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    print("Warning: SHAP not available. Install with: pip install shap")
    SHAP_AVAILABLE = False

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Import system components
from portfolio_system import SystemConfig, DataPipeline


In [ ]:
# Initialize configuration and data pipeline
config = SystemConfig()
pipeline = DataPipeline(config)

# Load and engineer features
print("Loading weekly data...")
weekly_data = pipeline.load_data(use_cache=True)

print("\nEngineering features...")
featured_data = pipeline.engineer_features(weekly_data)

# Get feature columns
exclude_cols = ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume',
               'dividends', 'stock splits', 'stock_splits', 'adj close', 'adj_close']
feature_cols = [col for col in featured_data.columns if col not in exclude_cols]

print(f"\n✓ Total features: {len(feature_cols)}")
print(f"✓ Data shape: {featured_data.shape}")
print(f"✓ Date range: {featured_data['date'].min()} to {featured_data['date'].max()}")


## 2. Feature Export and Basic Statistics

### Purpose

This section performs initial feature quality assessment and exports features for external analysis. It identifies:
- Features with high missing data rates (>10%)
- Features with excessive zero values (>50%)
- Features with zero variance (constant values)
- Features ranked by variance (indicating information content)

### Quality Metrics

- **Missing Percentage**: Proportion of NaN values per feature
- **Zero Percentage**: Proportion of zero values per feature
- **Variance**: Measure of feature spread (higher variance = more information)

### Interpretation

- High variance features contain more information and are generally more useful
- Features with >10% missing data may need special handling
- Features with >50% zeros may indicate sparse or binary features
- Zero variance features provide no information and should be removed

### Output Files

- `data/all_features_export.csv`: Complete feature dataset for external analysis


In [ ]:
# Export features to CSV for analysis
feature_export = featured_data[['date', 'ticker'] + feature_cols].copy()
export_path = config.data_dir / 'all_features_export.csv'
feature_export.to_csv(export_path, index=False)
print(f"✓ Features exported to: {export_path}")

# Basic statistics for each feature
feature_stats = featured_data[feature_cols].describe().T
feature_stats['missing_pct'] = (featured_data[feature_cols].isna().sum() / len(featured_data)) * 100
feature_stats['zero_pct'] = ((featured_data[feature_cols] == 0).sum() / len(featured_data)) * 100
feature_stats['variance'] = featured_data[feature_cols].var()

print("\n📊 Feature Statistics Summary:")
print(f"  Features with >10% missing: {(feature_stats['missing_pct'] > 10).sum()}")
print(f"  Features with >50% zeros: {(feature_stats['zero_pct'] > 50).sum()}")
print(f"  Features with zero variance: {(feature_stats['variance'] == 0).sum()}")

# Display top features by variance
print("\nTop 20 Features by Variance:")
print(feature_stats.nlargest(20, 'variance')[['mean', 'std', 'variance', 'missing_pct', 'zero_pct']])


In [ ]:
featured_data.columns


## 3. Correlation Analysis

### Purpose

Identify redundant features by computing pairwise correlations. Highly correlated features (|r| > 0.95) provide redundant information and can be removed to reduce dimensionality without significant information loss.

### Methodology

1. **Data Preparation**: Use training split only to avoid data leakage
2. **Correlation Matrix**: Compute Pearson correlation coefficients for all feature pairs
3. **Redundancy Detection**: Identify pairs with |correlation| > 0.95
4. **Feature Removal Strategy**: For highly correlated pairs, remove the feature with lower variance (keeps more informative feature)

### Correlation Threshold Rationale

- **0.95 threshold**: Balances redundancy removal with information preservation
  - Lower threshold (e.g., 0.90) would remove more features but risk losing complementary information
  - Higher threshold (e.g., 0.98) would be too conservative and miss obvious redundancies
- **Absolute value**: Both positive and negative high correlations indicate redundancy

### Visualization

The correlation heatmap shows relationships between top features by variance. Red indicates positive correlation, blue indicates negative correlation, and white indicates low correlation.

### Output

- List of highly correlated feature pairs
- Features recommended for removal (lower variance in correlated pairs)


In [ ]:
# Prepare data for correlation analysis (use training split only)
# Using only training data prevents data leakage - correlations should be computed
# on data that the model will see during training, not validation/test sets
splits = pipeline.create_train_val_test_splits(featured_data)
train_data = splits['train'].copy()

# Fill any remaining NaN values with 0
# This ensures correlation computation doesn't fail due to missing values
train_features = train_data[feature_cols].fillna(0)

# Calculate correlation matrix using Pearson correlation coefficient
# Pearson correlation measures linear relationships between features
print("Computing correlation matrix...")
correlation_matrix = train_features.corr()

print(f"\n✓ Correlation matrix shape: {correlation_matrix.shape}")

# Find highly correlated feature pairs (|correlation| > 0.95)
# Threshold 0.95 chosen to balance redundancy removal with information preservation:
# - Lower (e.g., 0.90) would remove more features but risk losing complementary info
# - Higher (e.g., 0.98) would be too conservative and miss obvious redundancies
# Using absolute value to catch both positive and negative high correlations
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.95:  # Threshold: 0.95
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                corr_val
            ))

print(f"\n📊 Highly Correlated Feature Pairs (|r| > 0.95): {len(high_corr_pairs)}")
if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
    print(high_corr_df.head(20))


In [ ]:
# Visualize correlation matrix (sample of features for readability)
# Select top 30 features by variance for visualization
top_features = train_features.var().nlargest(30).index.tolist()
corr_sample = correlation_matrix.loc[top_features, top_features]

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_sample, dtype=bool))
sns.heatmap(corr_sample, mask=mask, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('Feature Correlation Matrix (Top 30 Features by Variance)', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Identify redundant features using correlation
# For each highly correlated pair, keep the feature with higher variance
# Rationale: Higher variance features contain more information, so we prefer
# to keep them over lower variance features that provide redundant information
features_to_remove_corr = set()

for feat1, feat2, corr_val in high_corr_pairs:
    var1 = train_features[feat1].var()
    var2 = train_features[feat2].var()
    
    # Remove the feature with lower variance (keeps more informative feature)
    if var1 < var2:
        features_to_remove_corr.add(feat1)
    else:
        features_to_remove_corr.add(feat2)

print(f"\n📊 Features to remove due to high correlation: {len(features_to_remove_corr)}")
print(f"Remaining features after correlation removal: {len(feature_cols) - len(features_to_remove_corr)}")

if features_to_remove_corr:
    print("\nFeatures to remove:")
    for feat in sorted(features_to_remove_corr):
        print(f"  - {feat}")

# Keep features that are not highly correlated
features_after_corr = [f for f in feature_cols if f not in features_to_remove_corr]


## 4. Principal Component Analysis (PCA)

### Purpose

Determine the intrinsic dimensionality of the feature space and identify redundant dimensions. PCA transforms features into orthogonal components that capture maximum variance.

### Methodology

1. **Data Preparation**: 
   - Remove highly correlated features (from previous step)
   - Remove zero-variance features
   - Standardize features (mean=0, std=1) to ensure equal scaling

2. **PCA Computation**: Fit PCA on all components to analyze explained variance

3. **Variance Analysis**: Calculate cumulative explained variance to determine:
   - Number of components for 90% variance (captures most information)
   - Number of components for 95% variance (high information retention)
   - Number of components for 99% variance (near-complete information)

### Variance Threshold Interpretation

- **90% variance**: Good balance between dimensionality reduction and information retention
- **95% variance**: Conservative reduction, retains most information
- **99% variance**: Minimal reduction, captures nearly all information

### Component Analysis

Each principal component is a linear combination of original features. The loadings show which features contribute most to each component:
- High positive/negative loadings indicate strong contribution
- Components are ordered by explained variance (first component explains most variance)

### Use Case

While PCA identifies dimensionality, we don't use PCA-transformed features directly. Instead, we use PCA insights to inform feature selection, ensuring we keep features that contribute to high-variance components.


In [ ]:
# Prepare data for PCA
# Remove highly correlated features first (from correlation analysis step)
features_for_pca = features_after_corr.copy()
X_pca = train_features[features_for_pca].fillna(0)

# Remove features with zero variance
# Zero variance features provide no information and cause numerical issues in PCA
zero_var_features = X_pca.columns[X_pca.var() == 0].tolist()
features_for_pca = [f for f in features_for_pca if f not in zero_var_features]
X_pca = train_features[features_for_pca].fillna(0)

# Standardize features (mean=0, std=1)
# Critical for PCA: features with larger scales would dominate the analysis
# StandardScaler ensures all features contribute equally to principal components
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_pca)

# Run PCA on all components
# We fit on all components to analyze the full variance structure
print(f"Running PCA on {len(features_for_pca)} features...")
pca = PCA()
pca.fit(X_scaled)

# Calculate cumulative explained variance
# This tells us how much variance is captured by the first N components
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Find number of components for 90%, 95%, 99% variance thresholds
# These thresholds help understand the intrinsic dimensionality:
# - 90%: Good balance between reduction and information retention
# - 95%: Conservative reduction, retains most information
# - 99%: Near-complete information capture
n_90 = np.argmax(cumulative_variance >= 0.90) + 1
n_95 = np.argmax(cumulative_variance >= 0.95) + 1
n_99 = np.argmax(cumulative_variance >= 0.99) + 1

print(f"\n📊 PCA Results:")
print(f"  Components for 90% variance: {n_90}")
print(f"  Components for 95% variance: {n_95}")
print(f"  Components for 99% variance: {n_99}")
print(f"  Total components: {len(pca.components_)}")


In [ ]:
# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
n_components_show = min(50, len(pca.explained_variance_ratio_))
axes[0].plot(range(1, n_components_show + 1),
             pca.explained_variance_ratio_[:n_components_show], 'o-', linewidth=2, markersize=4)
axes[0].axvline(x=n_90, color='green', linestyle='--', label=f'90%: {n_90} components')
axes[0].axvline(x=n_95, color='orange', linestyle='--', label=f'95%: {n_95} components')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot (First 50 Components)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative variance plot
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, linewidth=2)
axes[1].axhline(y=0.90, color='green', linestyle='--', label='90% variance')
axes[1].axhline(y=0.95, color='orange', linestyle='--', label='95% variance')
axes[1].axhline(y=0.99, color='red', linestyle='--', label='99% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Analyze feature loadings for top components
# Get top 10 components
n_top_components = min(10, len(pca.components_))

print(f"\n📊 Top {n_top_components} Principal Components:")
print("=" * 80)

for i in range(n_top_components):
    explained_var = pca.explained_variance_ratio_[i]
    component = pca.components_[i]
    
    # Get top 10 features contributing to this component
    top_indices = np.argsort(np.abs(component))[-10:][::-1]
    top_features = [features_for_pca[idx] for idx in top_indices]
    top_loadings = component[top_indices]
    
    print(f"\nComponent {i+1} (Explained Variance: {explained_var:.2%}):")
    for feat, loading in zip(top_features, top_loadings):
        print(f"  {feat:30s}: {loading:7.4f}")


## 5. SHAP Values for Feature Importance

### Purpose

Rank features by their importance for predicting future returns using SHAP (SHapley Additive exPlanations) values. SHAP provides model-agnostic feature importance based on game theory.

### Methodology

1. **Target Variable**: Future 1-week log returns
   - Calculated as: log(next_week_close / current_week_close)
   - Shifted by -1 to get forward-looking returns
   - Grouped by ticker to ensure proper time ordering

2. **Surrogate Model**: Random Forest Regressor
   - Why Random Forest: Handles non-linear relationships, provides feature importance, works well with SHAP TreeExplainer
   - Parameters: 100 trees, max_depth=10, min_samples_split=20 (prevents overfitting)
   - Training sample: Up to 5000 samples (for computational efficiency)

3. **SHAP Computation**:
   - Uses TreeExplainer (optimized for tree-based models)
   - Computes SHAP values for 500 samples (SHAP is computationally expensive)
   - Calculates mean absolute SHAP values for feature ranking

### SHAP Values Interpretation

- **SHAP Value**: Contribution of a feature to the prediction for a specific sample
- **Mean |SHAP|**: Average absolute contribution across samples
- **Higher mean |SHAP|**: Feature has stronger predictive power

### Fallback Method

If SHAP is not available, falls back to Random Forest feature importances (Gini importance), which measures how much each feature contributes to reducing impurity in the trees.

### Computational Considerations

- SHAP computation is O(n_samples × n_features × n_trees), so we use a subset
- Random Forest training is faster, so we use a larger sample (5000)
- Trade-off: Larger samples = better estimates but longer computation time


In [ ]:
# Prepare target variable: future returns (1-week ahead)
# For each ticker, calculate next week's return
# Sorting ensures proper time ordering before shifting
train_data_sorted = train_data.sort_values(['ticker', 'date']).copy()

# Calculate forward returns from close prices (log returns)
# Note: log_returns column is not available in featured_data after engineering,
# so we calculate it directly from close prices
# Log returns: log(P_t+1 / P_t) = log(P_t+1) - log(P_t)
# Groupby('ticker') ensures we shift within each ticker's time series
train_data_sorted['next_close'] = train_data_sorted.groupby('ticker')['close'].shift(-1)
train_data_sorted['future_return'] = np.log(train_data_sorted['next_close'] / train_data_sorted['close'])
train_data_sorted = train_data_sorted.drop(columns=['next_close'])

# Remove rows with missing future returns
# Last week for each ticker will have NaN future_return (no next week available)
train_data_clean = train_data_sorted.dropna(subset=['future_return']).copy()

# Prepare features and target for SHAP analysis
# Use features after correlation removal and PCA preparation
X_shap = train_data_clean[features_for_pca].fillna(0)
y_shap = train_data_clean['future_return']

print(f"\n📊 SHAP Analysis Setup:")
print(f"  Features: {X_shap.shape[1]}")
print(f"  Samples: {X_shap.shape[0]}")
print(f"  Target: Future 1-week return")


In [ ]:
# Train a Random Forest model as surrogate for SHAP
# Random Forest chosen because:
# 1. Handles non-linear relationships well
# 2. Works efficiently with SHAP TreeExplainer
# 3. Provides feature importance as fallback if SHAP unavailable
# Use a sample for faster computation (SHAP is computationally expensive)
sample_size = min(5000, len(X_shap))  # Limit to 5000 samples for efficiency
sample_indices = np.random.choice(len(X_shap), sample_size, replace=False)
X_sample = X_shap.iloc[sample_indices]
y_sample = y_shap.iloc[sample_indices]

print(f"\nTraining Random Forest on {sample_size} samples...")
rf_model = RandomForestRegressor(
    n_estimators=100,        # Number of trees (more = better but slower)
    max_depth=10,             # Limit tree depth to prevent overfitting
    min_samples_split=20,    # Minimum samples to split (prevents overfitting)
    random_state=42,         # For reproducibility
    n_jobs=-1                # Use all CPU cores
)

rf_model.fit(X_sample, y_sample)
print(f"✓ Model trained. R² score: {rf_model.score(X_sample, y_sample):.4f}")


In [ ]:
# Calculate SHAP values (if available)
if SHAP_AVAILABLE:
    # Use a smaller sample for SHAP computation (it's computationally expensive)
    # SHAP complexity: O(n_samples × n_features × n_trees)
    # Using 500 samples balances accuracy with computation time
    shap_sample_size = min(500, len(X_sample))
    shap_indices = np.random.choice(len(X_sample), shap_sample_size, replace=False)
    X_shap_sample = X_sample.iloc[shap_indices]
    
    print(f"\nComputing SHAP values for {shap_sample_size} samples...")
    print("This may take a few minutes...")
    
    # Create SHAP explainer using TreeExplainer (optimized for tree models)
    # TreeExplainer is faster than KernelExplainer for tree-based models
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_shap_sample)
    
    print("✓ SHAP values computed")
    
    # Calculate mean absolute SHAP values for feature importance ranking
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Create feature importance dataframe
    shap_importance = pd.DataFrame({
        'feature': features_for_pca,
        'mean_abs_shap': mean_abs_shap,
        'rank': range(1, len(features_for_pca) + 1)
    }).sort_values('mean_abs_shap', ascending=False)
    
    shap_importance['rank'] = range(1, len(shap_importance) + 1)
    
    print("\n📊 Top 30 Features by SHAP Importance:")
    print("=" * 80)
    print(shap_importance.head(30).to_string(index=False))
else:
    # Fallback: use Random Forest feature importances
    print("\nUsing Random Forest feature importances (SHAP not available)...")
    rf_importance = pd.DataFrame({
        'feature': features_for_pca,
        'importance': rf_model.feature_importances_,
        'rank': range(1, len(features_for_pca) + 1)
    }).sort_values('importance', ascending=False)
    
    rf_importance['rank'] = range(1, len(rf_importance) + 1)
    shap_importance = rf_importance.rename(columns={'importance': 'mean_abs_shap'})
    
    print("\n📊 Top 30 Features by Random Forest Importance:")
    print("=" * 80)
    print(shap_importance.head(30).to_string(index=False))


In [ ]:
# Visualize feature importance
plt.figure(figsize=(12, 10))
top_n = 30
top_shap = shap_importance.head(top_n)

plt.barh(range(len(top_shap)), top_shap['mean_abs_shap'], color='steelblue')
plt.yticks(range(len(top_shap)), top_shap['feature'])
plt.xlabel('Mean |SHAP Value|' if SHAP_AVAILABLE else 'Feature Importance', fontsize=12)
plt.title(f'Top {top_n} Features by {"SHAP" if SHAP_AVAILABLE else "Random Forest"} Importance', fontsize=14, pad=20)
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 6. Feature Selection: Combining Methods

### Purpose

Combine insights from correlation analysis, PCA, and SHAP to select 30-40 core features that:
- Maximize predictive power (high SHAP importance)
- Minimize redundancy (low correlation)
- Ensure diversity across feature categories

### Selection Strategy

1. **Start with Top SHAP Features**: Select top N features by SHAP importance (target: 35 features)

2. **Categorize Features**: Group features into categories to ensure diversity:
   - **Returns**: Historical return features (1w, 2w, 4w, 8w, 26w)
   - **Volatility**: Volatility measures across different windows
   - **SMA**: Simple moving average features
   - **Price Ratios**: Price-to-SMA ratios, high-low ratios, close-open ratios
   - **Volume**: Volume change and volume ratios
   - **Technical**: Technical indicators (RSI, MACD, Bollinger Bands, ATR)
   - **Sharpe**: Risk-adjusted return measures
   - **Regime**: Market regime indicators (trend, volatility regimes, beta)
   - **Relative**: Cross-sectional relative features (relative returns, ranks)
   - **Other**: Additional features (drawdowns, skewness, kurtosis)

3. **Diversity Check**: Ensure representation across categories (prevents over-reliance on one feature type)

4. **Final Selection**: If fewer than target features, add more from importance ranking

### Rationale for 30-40 Features

- **Too few (<20)**: May miss important predictive signals
- **Too many (>50)**: Increases overfitting risk, slows training, adds noise
- **30-40**: Balance between information content and model complexity

### Output

- List of selected features with SHAP importance scores
- Feature distribution across categories


In [ ]:
# Combine all methods to select 30-40 core features
# Strategy:
# 1. Remove highly correlated features (already done in correlation analysis)
# 2. Keep top features by SHAP/RF importance (from SHAP analysis)
# 3. Ensure diversity across feature categories (prevents over-reliance on one type)

# Target: 30-40 features balances information content with model complexity
# - Too few (<20): May miss important predictive signals
# - Too many (>50): Increases overfitting risk, slows training, adds noise
target_n_features = 35  # Target 30-40 features

# Start with top SHAP features
top_shap_features = shap_importance.head(target_n_features)['feature'].tolist()

print(f"\n📊 Feature Selection Strategy:")
print(f"  Target: {target_n_features} features")
print(f"  Top importance features: {len(top_shap_features)}")

# Categorize features to ensure diversity
feature_categories = {
    'returns': [f for f in top_shap_features if 'returns' in f.lower() or 'return' in f.lower()],
    'volatility': [f for f in top_shap_features if 'volatility' in f.lower() or 'vol' in f.lower()],
    'sma': [f for f in top_shap_features if 'sma' in f.lower()],
    'price_ratios': [f for f in top_shap_features if 'price_to_sma' in f.lower() or 'ratio' in f.lower()],
    'volume': [f for f in top_shap_features if 'volume' in f.lower()],
    'technical': [f for f in top_shap_features if any(x in f.lower() for x in ['rsi', 'macd', 'bb', 'atr'])],
    'sharpe': [f for f in top_shap_features if 'sharpe' in f.lower()],
    'regime': [f for f in top_shap_features if any(x in f.lower() for x in ['regime', 'trend', 'beta'])],
    'relative': [f for f in top_shap_features if 'relative' in f.lower() or 'rank' in f.lower()],
    'other': []
}

# Add remaining features to 'other'
all_categorized = sum(feature_categories.values(), [])
feature_categories['other'] = [f for f in top_shap_features if f not in all_categorized]

print("\n📊 Feature Distribution by Category:")
for category, features in feature_categories.items():
    if features:
        print(f"  {category:15s}: {len(features):2d} features")
        for feat in features[:5]:  # Show first 5
            print(f"    - {feat}")
        if len(features) > 5:
            print(f"    ... and {len(features) - 5} more")


In [ ]:
# Final feature selection: ensure we have diverse representation
# If we have fewer than target, add more from importance ranking
selected_features = top_shap_features.copy()

if len(selected_features) < target_n_features:
    # Add more features from importance ranking
    remaining_features = shap_importance[~shap_importance['feature'].isin(selected_features)]
    n_needed = target_n_features - len(selected_features)
    selected_features.extend(remaining_features.head(n_needed)['feature'].tolist())

# Ensure we don't exceed target
selected_features = selected_features[:target_n_features]

print(f"\n✓ Final Selected Features: {len(selected_features)}")
print("\nSelected Features:")
for i, feat in enumerate(selected_features, 1):
    importance_val = shap_importance[shap_importance['feature'] == feat]['mean_abs_shap'].values[0]
    print(f"  {i:2d}. {feat:35s} ({'SHAP' if SHAP_AVAILABLE else 'RF'}: {importance_val:.6f})")


In [ ]:
# Verify selected features don't have high correlation
selected_corr = train_features[selected_features].corr()

# Count high correlations in selected set
high_corr_count = 0
for i in range(len(selected_corr.columns)):
    for j in range(i+1, len(selected_corr.columns)):
        if abs(selected_corr.iloc[i, j]) > 0.90:
            high_corr_count += 1

print(f"\n📊 Selected Features Validation:")
print(f"  Total features: {len(selected_features)}")
print(f"  High correlations (|r| > 0.90): {high_corr_count}")
print(f"  Average correlation: {selected_corr.values[np.triu_indices_from(selected_corr.values, k=1)].mean():.4f}")

if high_corr_count > 0:
    print("\n⚠️ Warning: Some selected features are highly correlated")
else:
    print("\n✓ Good: No highly correlated features in selected set")


## 7. Export Selected Features

### Purpose

Export the selected feature list in formats suitable for use in the main portfolio system.

### Export Formats

1. **CSV Format** (`selected_features.csv`):
   - Columns: feature name, importance score, rank
   - Useful for analysis in spreadsheet tools or other languages
   - Includes importance scores for reference

2. **JSON Format** (`selected_features.json`):
   - Simple list of feature names
   - Easy to import in Python: `json.load(open('selected_features.json'))`
   - Used by DataPipeline to filter features automatically

### Usage in Portfolio System

The `DataPipeline` class automatically loads `selected_features.json` if available and filters features accordingly. This ensures consistency between exploration and training phases.

### Summary Statistics

Reports the reduction in feature count:
- Original features: All engineered features
- After correlation removal: Features remaining after removing highly correlated pairs
- Final selected features: Features selected by the multi-method approach


In [ ]:
# Export selected features list
selected_features_df = pd.DataFrame({
    'feature': selected_features,
    'importance': [shap_importance[shap_importance['feature'] == f]['mean_abs_shap'].values[0] 
                   for f in selected_features],
    'rank': [shap_importance[shap_importance['feature'] == f]['rank'].values[0] 
             for f in selected_features]
})

export_path = config.data_dir / 'selected_features.csv'
selected_features_df.to_csv(export_path, index=False)
print(f"✓ Selected features exported to: {export_path}")

# Also save as Python list for easy import
import json
features_json_path = config.data_dir / 'selected_features.json'
with open(features_json_path, 'w') as f:
    json.dump(selected_features, f, indent=2)
print(f"✓ Selected features list exported to: {features_json_path}")

# Summary statistics
print(f"\n📊 Feature Selection Summary:")
print(f"  Original features: {len(feature_cols)}")
print(f"  After correlation removal: {len(features_for_pca)}")
print(f"  Final selected features: {len(selected_features)}")
print(f"  Reduction: {len(feature_cols) - len(selected_features)} features ({100*(len(feature_cols) - len(selected_features))/len(feature_cols):.1f}% reduction)")


## 8. Feature Reduction Impact Analysis

### Purpose

Validate that selected features maintain predictive performance compared to the full feature set. This ensures we're not losing critical information through feature selection.

### Methodology

1. **Data Preparation**:
   - Use same target variable (future returns) for both models
   - Align indices to ensure consistent comparison
   - Split into train/validation sets (80/20)

2. **Model Training**:
   - Train Random Forest on full feature set
   - Train Random Forest on selected feature set
   - Same hyperparameters for fair comparison

3. **Performance Metrics**:
   - R² score on validation set
   - Performance retention: (selected_score / full_score) × 100%

### Interpretation

- **Performance Retention >95%**: Excellent - selected features retain almost all predictive power
- **Performance Retention 90-95%**: Good - acceptable trade-off for reduced complexity
- **Performance Retention <90%**: Warning - may have removed important features

### Important Notes

- Negative R² scores indicate the model performs worse than predicting the mean
- This is common in financial prediction tasks due to market efficiency
- The comparison is relative: if both models have negative R², we compare which is less negative
- The goal is feature selection, not necessarily achieving positive R²


In [ ]:
# Compare model performance with full vs selected features
# This validates that feature selection doesn't significantly harm predictive power
# Train simple models to compare (using same hyperparameters for fair comparison)

# Full feature set (all original features)
X_full = train_features[feature_cols].fillna(0)
y_full = train_data_clean['future_return']

# Selected feature set (features chosen by our selection process)
X_selected = train_features[selected_features].fillna(0)
y_selected = train_data_clean['future_return']

# Align targets to ensure consistent comparison
# Both models use the same target variable and same samples
common_idx = X_full.index.intersection(y_full.index)
X_full = X_full.loc[common_idx]
X_selected = X_selected.loc[common_idx]
y_full = y_full.loc[common_idx]
y_selected = y_full  # Same target for both models

# Split for validation (80/20 split)
# Using same random_state ensures both models see same train/val split
X_full_train, X_full_val, y_full_train, y_full_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

X_sel_train, X_sel_val, y_sel_train, y_sel_val = train_test_split(
    X_selected, y_selected, test_size=0.2, random_state=42
)

# Train models with identical hyperparameters for fair comparison
# Using fewer trees (50) than SHAP model (100) for faster computation
print("\nTraining models for comparison...")
rf_full = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_full.fit(X_full_train, y_full_train)

rf_selected = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_selected.fit(X_sel_train, y_sel_train)

# Evaluate
score_full = rf_full.score(X_full_val, y_full_val)
score_selected = rf_selected.score(X_sel_val, y_sel_val)

print(f"\n📊 Model Performance Comparison:")
print(f"  Full features ({len(feature_cols)}): R² = {score_full:.4f}")
print(f"  Selected features ({len(selected_features)}): R² = {score_selected:.4f}")
print(f"  Performance retention: {100 * score_selected / score_full:.1f}%")

if score_selected >= score_full * 0.95:
    print("\n✓ Excellent: Selected features retain >95% of performance")
elif score_selected >= score_full * 0.90:
    print("\n✓ Good: Selected features retain >90% of performance")
else:
    print("\n⚠️ Warning: Significant performance loss with selected features")


## 9. Recommendations

### Purpose

Provide actionable recommendations for implementing the selected features in the portfolio management system and next steps for model optimization.

### Implementation Steps

1. **Update Portfolio System**: Use `selected_features.json` in `portfolio_system.py`
2. **Retrain Models**: Retrain RL agent with reduced feature set
3. **Monitor Performance**: Compare training stability and convergence
4. **Iterate**: Refine feature selection based on model performance

### Feature Category Priorities

The recommendations show which feature categories are most represented in the selected set, helping understand what types of information the model relies on most.

### Next Steps

- **Model Training**: Use selected features for RL agent training
- **Performance Monitoring**: Track training metrics and validation performance
- **Feature Refinement**: Consider removing features with very low importance if model performance is poor
- **Regular Updates**: Re-run feature selection periodically as new data becomes available


In [ ]:
print("\n" + "=" * 80)
print("FEATURE SELECTION RECOMMENDATIONS")
print("=" * 80)

print("\n1. IMPLEMENTATION:")
print(f"   - Use the {len(selected_features)} selected features in portfolio_system.py")
print(f"   - Update feature_cols to use selected_features.json")
print(f"   - This reduces observation space from {len(feature_cols) * 7} to {len(selected_features) * 7} dimensions")

print("\n2. FEATURE CATEGORIES TO PRIORITIZE:")
for category, features in feature_categories.items():
    if features:
        print(f"   - {category.capitalize()}: {len(features)} features")

print("\n3. FEATURES TO REMOVE:")
removed_features = [f for f in feature_cols if f not in selected_features]
print(f"   - {len(removed_features)} features can be removed")
print(f"   - Top candidates: features with low importance and high correlation")

print("\n4. NEXT STEPS:")
print("   - Update portfolio_system.py to use selected features")
print("   - Retrain models with reduced feature set")
print("   - Compare performance: should maintain or improve due to reduced overfitting")
print("   - Monitor training stability and convergence")

print("\n" + "=" * 80)
